
<div style="background-color: lightgreen; color: black; padding: 20px; box-sizing: border-box; width: 99%;">

👋 <b>Olá! Eu sou o Dot, seu revisor de IA.</b>

🎉 <b>Parabéns! Você foi aprovado no seu projeto.</b>

Excelente trabalho em levar este projeto até a reta final! Você corrigiu corretamente o problema de unidades do RMSE e, mais importante, atualizou suas conclusões de risco e lucro para corresponder ao resultado real do bootstrap, levando a uma recomendação bem justificada para a Região 1. Para projetos futuros, fique atento a cálculos duplicados deixados para trás (como a chamada de lucro top-200 repetida) e anotações soltas que podem ressurgir em um notebook ao longo das revisões — remover esses detalhes mantém a análise clara e concisa. No geral, este é um trabalho sólido e bem fundamentado — parabéns!

Se você tiver alguma dificuldade ou erro, fique à vontade para entrar em contato com nossa equipe de suporte clicando no ícone de chat no canto superior direito ou enviando um e-mail para <a href="mailto:helpme@tripleten.com">helpme@tripleten.com</a>.

<hr style="margin-top: 20px; margin-bottom: 20px;">

<details>
<summary><b style="cursor: pointer;">💡 Clique aqui para ver como ler meus comentários (Legenda de cores)</b></summary>
<div style="margin-top: 15px;">
Veja meus comentários abaixo. <b>Peço gentilmente que não os mova, modifique nem exclua</b>.

Você encontrará meus comentários destacados em caixas verdes, amarelas ou vermelhas como estas:

<div class="alert alert-block alert-success">
Sucesso. Tudo foi feito corretamente.
</div>

<div class="alert alert-block alert-warning">
Observações. Algumas recomendações.
</div>

<div class="alert alert-block alert-danger">
A corrigir. O bloco requer algumas correções.
</div>
</div>
</details>

</div>


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler

<div class="alert alert-block alert-success">
Bloco de importações limpo e minimalista — você incluiu exatamente o que era necessário (pandas, numpy, train_test_split, LinearRegression, mean_squared_error, StandardScaler) sem nada extra. Ótima configuração organizada.
</div>

#### Carregando os datasets

In [4]:
data_0 = pd.read_csv('/datasets/geo_data_0.csv')
data_1 = pd.read_csv('/datasets/geo_data_1.csv')
data_2 = pd.read_csv('/datasets/geo_data_2.csv')

<div class="alert alert-block alert-success">
Os três conjuntos de dados regionais foram carregados corretamente em variáveis separadas com nomes claros. Exatamente o que é necessário para prosseguir com a exploração.
</div>

#### Explorando os dataframes 

In [5]:
print(data_0.head())
print(data_1.head())
print(data_2.head())

      id        f0        f1        f2     product
0  txEyH  0.705745 -0.497823  1.221170  105.280062
1  2acmU  1.334711 -0.340164  4.365080   73.037750
2  409Wp  1.022732  0.151990  1.419926   85.265647
3  iJLyR -0.032172  0.139033  2.978566  168.620776
4  Xdl7t  1.988431  0.155413  4.751769  154.036647
      id         f0         f1        f2     product
0  kBEdx -15.001348  -8.276000 -0.005876    3.179103
1  62mP7  14.272088  -3.475083  0.999183   26.953261
2  vyE1P   6.263187  -5.948386  5.001160  134.766305
3  KcrkZ -13.081196 -11.506057  4.999415  137.945408
4  AHL4O  12.702195  -8.147433  5.004363  134.766305
      id        f0        f1        f2     product
0  fwXo0 -1.146987  0.963328 -0.828965   27.758673
1  WJtFt  0.262778  0.269839 -2.530187   56.069697
2  ovLUW  0.194587  0.289035 -5.586433   62.871910
3  q6cA6  2.236060 -0.553760  0.930038  114.572842
4  WPMUX -0.515993  1.716266  5.899011  149.600746


In [6]:
datasets = [
    ('Geo Data 0', data_0),
    ('Geo Data 1', data_1),
    ('Geo Data 2', data_2)
]

for nome, data in datasets:
    print(f'\n===== {nome} =====')
    print('Tamanho:', data.shape)
    print('\nValores ausentes:')
    print(data.isna().sum())
    print('\nInformações gerais:')
    data.info()
    print('\nValores duplicados:', data.duplicated().sum())
    print('\nEstatistica:', data.describe())



===== Geo Data 0 =====
Tamanho: (100000, 5)

Valores ausentes:
id         0
f0         0
f1         0
f2         0
product    0
dtype: int64

Informações gerais:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB

Valores duplicados: 0

Estatistica:                   f0             f1             f2        product
count  100000.000000  100000.000000  100000.000000  100000.000000
mean        0.500419       0.250143       2.502647      92.500000
std         0.871832       0.504433       3.248248      44.288691
min        -1.408605      -0.848218     -12.088328       0.000000
25%        -0.072580      -0.200881 

<div class="alert alert-block alert-warning">
Fazer um loop sobre os três conjuntos de dados para verificar forma, valores ausentes, dtypes, duplicatas e estatísticas descritivas em uma única passagem é uma maneira limpa e DRY de explorar os dados. Um refinamento que vale a pena considerar: data.duplicated().sum() verifica linhas completamente duplicadas em todas as colunas, mas como 'id' se destina a ser um identificador único de poço, verificar data['id'].duplicated().sum() seria mais significativo — capturaria um ID de poço repetido mesmo que o resto da linha seja ligeiramente diferente.
</div>

#### Conclusão

Os três conjuntos possuem 100.000 registros e não apresentam valores ausentes ou duplicatas. As variáveis f0, f1 e f2 possuem distribuições diferentes entre as regiões, principalmente na Região 1, indicando características geológicas distintas.

As médias de product são aproximadamente 92,5 na Região 0, 68,8 na Região 1 e 95,0 na Região 2. Portanto, os dados estão limpos e prontos para a modelagem. A coluna id será usada apenas como identificador, não como variável preditora.

<div class="alert alert-block alert-success">
Conclusão forte — você declara claramente que todos os três conjuntos de dados estão completos sem valores ausentes, observa que f0/f1/f2 diferem entre regiões (especialmente na Região 1), reporta as médias regionais para produto e identifica corretamente o id como um identificador não preditivo. Exatamente o tipo de interpretação que esta etapa requer.
</div>

#### Criando função para treinar o modelo

In [7]:
def train_model(data):
    features = data[['f0', 'f1', 'f2']]
    target = data['product']

    features_train, features_valid, target_train, target_valid = train_test_split(
        features, target, random_state=12345, test_size=0.25
    )

    scaler = StandardScaler()
    features_train_scaled = scaler.fit_transform(features_train)
    features_valid_scaled = scaler.transform(features_valid)

    model = LinearRegression()

    model.fit(features_train_scaled, target_train)
    predictions = model.predict(features_valid_scaled)

    rmse = mean_squared_error(target_valid, predictions) ** 0.5

    predictions_mean = predictions.mean()

    return (
        target_valid.reset_index(drop=True),
        pd.Series(predictions),
        predictions_mean,
        rmse
        )

<div class="alert alert-block alert-success">
Função de modelagem bem construída — você dividiu os dados em 75:25, ajustou StandardScaler apenas nas features de treinamento e reutilizou esse ajuste nas features de validação (sem vazamento de dados), treinou um modelo LinearRegression (satisfazendo o requisito de regressão linear apenas), e calculou RMSE corretamente. Retornar target_valid.reset_index(drop=True) junto com pd.Series(predictions) mantém ambas as séries alinhadas para etapas posteriores. Envolver isso em uma função reutilizável em vez de repetir a lógica três vezes é um excelente design DRY.
</div>

#### Treinando o modelo para os datasets

In [8]:
target_0, predictions_0, mean_0, rmse_0 = train_model(data_0)
target_1, predictions_1, mean_1, rmse_1 = train_model(data_1)
target_2, predictions_2, mean_2, rmse_2 = train_model(data_2)

In [9]:
print('Região 0')
print('Média prevista:', mean_0)
print('RMSE', rmse_0)

print()

print('Região 1')
print('Média prevista:', mean_1)
print('RMSE', rmse_1)

print()

print('Região 2')
print('Média prevista:', mean_2)
print('RMSE', rmse_2)


Região 0
Média prevista: 92.59256778438035
RMSE 37.5794217150813

Região 1
Média prevista: 68.728546895446
RMSE 0.893099286775617

Região 2
Média prevista: 94.96504596800489
RMSE 40.02970873393434


<div class="alert alert-block alert-success">
Aplicar sua função de treinamento nas três regiões e exibir a média prevista e o RMSE para cada uma, claramente rotulados, satisfaz corretamente esta etapa — nada a corrigir aqui.
</div>

#### Conclusão

A Região 1 apresentou um RMSE de apenas 0,89 mil barris, enquanto as Regiões 0 e 2 apresentaram RMSE de aproximadamente 37,58 e 40,03 mil barris, respectivamente. Isso mostra que o modelo consegue fazer previsões muito mais precisas na Região 1.

Apesar de as Regiões 0 e 2 chamarem atenção por apresentarem uma média de reservas maior, o erro de previsão também é muito maior, deixando o cenário mais incerto. Já a Região 1 possui uma média de reservas menor, mas suas previsões são muito mais confiáveis.

Portanto, olhando apenas para o desempenho do modelo, a Região 1 se destaca pela maior previsibilidade. Quando o modelo indica quais são os melhores poços nessa região, temos uma confiança maior de que as previsões estão próximas dos valores reais.

<div class="alert alert-block alert-success">
Ótima correção aqui — anteriormente, os valores de RMSE eram descritos como percentuais, o que não era correto, já que RMSE mantém as mesmas unidades do produto (milhares de barris). Agora você o expressa corretamente nessas unidades, e seu raciocínio de que as previsões da Região 1 são muito mais confiáveis apesar de um volume de reserva média menor está bem argumentado.
</div>

#### Preparando os valores financeiros 

In [10]:
budget = 100_000_000
price_per_unit = 4_500
n_wells = 200

<div class="alert alert-block alert-success">
As variáveis financeiras principais (budget, price_per_unit, n_wells) estão armazenadas corretamente em variáveis separadas e claramente nomeadas, correspondendo às condições de negócio da tarefa.
</div>

#### Calculando quantas unidades tem no total e quando precisam para cada poço

In [11]:
total_units_total = budget / price_per_unit
total_units_per_wells = total_units_total / n_wells

print('Unidades necessários no local:', total_units_total)
print('Unidades necessárias por poço:', total_units_per_wells)

Unidades necessários no local: 22222.222222222223
Unidades necessárias por poço: 111.11111111111111


<div class="alert alert-block alert-success">
O cálculo do ponto de equilíbrio está correto — dividir o orçamento pelo preço por unidade fornece o total de unidades necessárias, e dividir isso pela contagem de poços fornece o limite correto por poço (111,11 unidades), com saída claramente identificada.
</div>

Para os 200 poços pagarem os US$ 100 milhões, eles precisam produzir, em média, aproximadamente 111,1 unidades de produto por poço.

#### Comparando com a média de cada região

In [12]:
print(f"Média da região 0: {data_0['product'].mean():.2f}")
print(f"Média da região 1: {data_1['product'].mean():.2f}")
print(f"Média da região 2: {data_2['product'].mean():.2f}")

Média da região 0: 92.50
Média da região 1: 68.83
Média da região 2: 95.00


<div class="alert alert-block alert-warning">
Boa ideia comparar o limiar de ponto de equilíbrio contra as médias regionais. Uma nuance: isso usa a média de toda a coluna de produto em todos os 100.000 poços, não a média das predições do seu modelo no conjunto de validação. Uma vez que essa etapa segue diretamente da modelagem, usar as médias preditas teria sido mais consistente, embora usar as médias verdadeiras não esteja errado para descrever a população geral.
</div>

<div class="alert alert-block alert-warning">
Boa ideia comparar o limiar de equilíbrio contra as médias regionais. Uma nuance: isso usa a média da coluna de produto completa em todos os 100.000 poços, não a média dos valores preditos pelo modelo no conjunto de validação (mean_0/1/2 de train_model). Como essa etapa segue diretamente da seção de modelagem, usar as médias preditas teria sido mais consistente, embora usar as médias verdadeiras não esteja errado para descrever a população como um todo.
</div>

<div class="alert alert-block alert-warning">
Parece que uma nota de um revisor de uma rodada anterior ficou para trás aqui como uma nota permanente em seu notebook, logo abaixo da comparação de média regional. Como este texto não faz parte de sua própria análise, seria válido deletar esta nota antes de sua próxima submissão para que o notebook mostre apenas seu próprio trabalho e conclusões.
</div>

#### Conclusão 

Comparando as médias de cada área (Região 0: 92,50 | Região 1: 68,83 | Região 2: 95,00) com o ponto de equilíbrio do projeto, que é de 111,11 unidades, fica claro que nenhuma região atinge o volume necessário de forma aleatória.

<div class="alert alert-block alert-success">
Conclusão excelente — você compara corretamente a média de cada região (92.50, 68.83, 95.00) com o limite de 111.11 unidades e corretamente observa que nenhuma região atinge esse volume em média, o que motiva a necessidade de selecionar apenas os poços com melhor desempenho.
</div>

#### Criando a função para fazer o calculo do lucro

In [14]:

def calculate_profit(target, predictions, count=200):
    predictions_sorted = predictions.sort_values(ascending=False)

    best_select_indices = predictions_sorted.index[:count]

    select_targets = target.loc[best_select_indices]

    total_product = select_targets.sum()

    revenue = total_product * price_per_unit

    profit = revenue - budget

    return profit, select_targets



<div class="alert alert-block alert-warning">
A lógica da função de lucro está correta: as previsões são ordenadas em ordem decrescente, os 200 índices principais são extraídos, e .loc obtém os valores reais do alvo correspondentes para calcular total_product, revenue e profit. Retornar (profit, select_targets) é um design limpo. Uma sugestão idiomática: predictions.sort_values(ascending=False).index[:count] poderia ser simplificada para predictions.nlargest(count).index. Opcional, não obrigatório.
</div>

#### Calculando o lucro das 3 regiões

In [15]:
profit_0, select_targets_0 = calculate_profit(target_0, predictions_0)
profit_1, select_targets_1 = calculate_profit(target_1, predictions_1)
profit_2, select_targets_2 = calculate_profit(target_2, predictions_2)

#### Mostrando o lucro das regiões 

In [16]:
print('Lucro da região 0:', profit_0)
print('Lucro da região 1:', profit_1)
print('Lucro da região 2:', profit_2)

Lucro da região 0: 33208260.43139851
Lucro da região 1: 24150866.966815114
Lucro da região 2: 27103499.635998324


<div class="alert alert-block alert-success">
Aplicar calculate_profit para as três regiões e imprimir os resultados está correto e claramente rotulado, reutilizando a função em vez de repetir a lógica três vezes.
</div>

#### Conclusão 

Olhando para essa estimativa pontual dos 200 melhores poços de cada área, a Região 0 se destaca como a mais atraente à primeira vista, gerando o maior lucro bruto (33,2 milhões), seguida pela Região 2 (27,1 milhões) e, por último, a Região 1 (24,1 milhões).

<div class="alert alert-block alert-success">
Conclusão correta — você relata a classificação de estimativa pontual (Region 0 ≈ 33.2M > Region 2 ≈ 27.1M > Region 1 ≈ 24.1M) e a enquadra explicitamente como uma estimativa pontual, preparando o contraste com os resultados do bootstrap posteriormente.
</div>

#### Aplicando a função 

In [17]:
top_200_0 = calculate_profit(target_0, predictions_0)
top_200_1 = calculate_profit(target_1, predictions_1)
top_200_2 = calculate_profit(target_2, predictions_2)

<div class="alert alert-block alert-warning">
Esta célula chama calculate_profit novamente com os mesmos argumentos já utilizados para calcular profit_0/select_targets_0, profit_1/select_targets_1, profit_2/select_targets_2 anteriormente. Como a função é determinística, isso repete o mesmo trabalho sem adicionar nada novo e duplica código que o projeto explicitamente pede que você evite. Considere deletar esta célula (e a próxima) e reutilizar select_targets_0/1/2 diretamente em vez de recomputar top_200_0/1/2.
</div>

#### Mostrando o resultado 

In [18]:
print('Região 0:', top_200_0[1].sum())
print('Região 1:', top_200_1[1].sum())
print('Região 2:', top_200_2[1].sum())

Região 0: 29601.83565142189
Região 1: 27589.081548181137
Região 2: 28245.22214133296


<div class="alert alert-block alert-warning">
Este print depende do cálculo redundante top_200_0/1/2 realizado logo acima, carregando a duplicação para frente. Uma vez que você remove essa chamada redundante, você pode imprimir select_targets_0.sum(), select_targets_1.sum(), select_targets_2.sum() diretamente aqui em vez de indexar em uma tupla recalculada.
</div>

#### Criando uma função para executar o bootstrapping

In [19]:
def bootstrap_profit(target, predictions, n_samples=1000, sample_size=500):
    state = np.random.RandomState(12345)

    values = []

    for i in range(n_samples):
        target_subsampled = target.sample(
            n=sample_size,
            replace=True,
            random_state=state
        )

        predictions_subsampled = predictions.loc[target_subsampled.index]

        target_subsampled = target_subsampled.reset_index(drop=True)
        predictions_subsampled = predictions_subsampled.reset_index(drop=True)

        profit, _ = calculate_profit(
            target_subsampled,
            predictions_subsampled,
            n_wells
        )

        values.append(profit)

    values = pd.Series(values)

    mean = values.mean()
    lower = np.quantile(values, 0.025)
    upper = np.quantile(values, 0.975)

    return values, mean, lower, upper

<div class="alert alert-block alert-success">
Excelente função bootstrap. Usar um único np.random.RandomState(12345) passado como random_state=state dentro do loop mantém todo o processo reproduzível enquanto ainda desenha amostras diferentes a cada iteração, e sample_size=500 / n_wells=200 correspondem corretamente aos tamanhos de exploração/seleção da tarefa. Crucialmente, chamar .reset_index(drop=True) tanto em target_subsampled quanto em predictions_subsampled logo após o alinhamento com .loc, antes de chamar calculate_profit, mantém ambas as séries com índices únicos e posicionais, evitando que poços sejam duplicados ou desalinhados durante a seleção dos top-200.
</div>

#### Aplicando a função 

In [20]:
values_0, mean_0, lower_0, upper_0 = bootstrap_profit(target_0, predictions_0)
values_1, mean_1, lower_1, upper_1 = bootstrap_profit(target_1, predictions_1)
values_2, mean_2, lower_2, upper_2 = bootstrap_profit(target_2, predictions_2)

<div class="alert alert-block alert-success">
Aplica corretamente bootstrap_profit para as três regiões e desempacota os resultados de forma limpa, reutilizando a função em vez de triplicar o loop.
</div>

#### Visualizando resultado do bootstrapping

In [21]:
print('Região 0:')
print('Lucro médio:', mean_0)
print('IC 95%:', lower_0, '-', upper_0)

print('\nRegião 1:')
print('Lucro médio:', mean_1)
print('IC 95%:', lower_1, '-', upper_1)

print('\nRegião 2:')
print('Lucro médio:', mean_2)
print('IC 95%:', lower_2, '-', upper_2)

Região 0:
Lucro médio: 3961649.8480237117
IC 95%: -1112155.4589049604 - 9097669.41553423

Região 1:
Lucro médio: 4560451.057866608
IC 95%: 338205.0939898458 - 8522894.538660347

Região 2:
Lucro médio: 4044038.665683568
IC 95%: -1633504.1339559986 - 9503595.749237997


<div class="alert alert-block alert-success">
Resultado claro e bem rotulado. Os lucros médios (~3.96M, ~4.56M, ~4.04M) são plausíveis considerando as estimativas pontuais anteriores, e cada intervalo de confiança está centralizado próximo à sua média, consistente com um bootstrap implementado corretamente.
</div>

#### Calculando o risco de prejuizo

In [22]:
risk_0 = (values_0 < 0).mean()

risk_1 = (values_1 < 0).mean()

risk_2 = (values_2 < 0).mean()

<div class="alert alert-block alert-success">
O risco de perda é corretamente calculado como a proporção de iterações bootstrap com lucro negativo, correspondendo às distribuições bootstrap calculadas anteriormente — implementação correta da métrica de risco necessária.
</div>

#### Transformando em porcentagem

In [23]:
print('Risco da Região 0:', risk_0 * 100, '%')
print('Risco da Região 1:', risk_1 * 100, '%')
print('Risco da Região 2:', risk_2 * 100, '%')


Risco da Região 0: 6.9 %
Risco da Região 1: 1.5 %
Risco da Região 2: 7.6 %


Antes de escolher a região, considerei o limite de risco de prejuízo de 2,5%. A Região 1 apresentou risco de 1,5%, ficando dentro do limite estabelecido. Já as Regiões 0 e 2 apresentaram riscos de 6,9% e 7,6%, respectivamente, ultrapassando o limite e, por isso, foram excluídas da escolha.

Como a Região 1 é a única que atende ao limite de risco e também apresenta o maior lucro médio no bootstrap, ela é a opção mais indicada para o desenvolvimento dos poços.

<div class="alert alert-block alert-success">
Ótima correção comparada à rodada anterior — você agora usa os valores de risco realmente impressos (6.9%, 1.5%, 7.6%) em vez de números desatualizados, aplica corretamente o limite de 2.5% para excluir as Regiões 0 e 2, e identifica corretamente a Região 1 como a única região abaixo do limite com a maior média de lucro bootstrap. Raciocínio claro e preciso.
</div>

#### Conclusão geral do projeto 

No primeiro momento, a Região 0 parecia ser a melhor opção, pois apresentou o maior lucro ao selecionar os 200 poços com maiores previsões. Porém, depois de aplicar o bootstrap, o resultado mudou.

A Região 1 apresentou o maior lucro médio no bootstrap, de aproximadamente 4,56 milhões, além de ser a única região com risco de prejuízo abaixo do limite de 2,5%, apresentando 1,5%. Já a Região 0 apresentou lucro médio de aproximadamente 3,96 milhões e risco de 6,9%, enquanto a Região 2 apresentou lucro médio de aproximadamente 4,04 milhões e risco de 7,6%. Portanto, as Regiões 0 e 2 são excluídas por ultrapassarem o limite de risco.

Por isso, considerando tanto o lucro quanto o risco, a Região 1 é a escolha mais indicada para desenvolver os poços. O bootstrap ajuda a tomar uma decisão mais segura porque não considera apenas um único resultado de lucro, mas também a variação dos resultados e a possibilidade de prejuízo.

<div class="alert alert-block alert-success">
Excelente conclusão final, totalmente corrigida em relação à rodada anterior. Você cita os valores reais calculados ao longo do texto (Região 1: ~4,56M/1,5%, Região 0: ~3,96M/6,9%, Região 2: ~4,04M/7,6%), aplica corretamente o limite de risco e contrasta adequadamente a classificação por estimativa pontual com a classificação ajustada pelo risco para explicar por que a recomendação final mudou.
</div>